# scRNA-seq pathway enrichment analysis — malignant cells (single cell GSEA)

**Goal:** Identify pathways enriched in MHC II high vs low malignant cells in the
Salcher scRNA-seq atlas using GSEA. Results are compared to the CosMx spatial
transcriptomics pathway analysis to identify concordantly enriched pathways across
both modalities.

**Input:** `cancer_cells_mhc2_classification.h5ad` — Salcher atlas malignant cells
with MHC II high/low classification; `data/cosmx_pathway_annotations.csv` — 
NanoString CosMx pathway gene set annotations.

**Output:** `outputs/tables/figure3/scrna_gsea_hallmark_sngle_cell.csv`;
`outputs/tables/figure3/scrna_gsea_cosmx_pathways_sngle_cell.csv``

## Setup

In [ ]:
from pathlib import Path
import yaml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

import anndata as ad
import scanpy as sc
import decoupler as dc

# figure settings
sns.set(font_scale=1.8)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
cmap_low_high = ['#462255FF', '#FF8811FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']
cmap_high_low = ['#FF8811FF', '#462255FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']

In [ ]:
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])

# inputs
# the scRNA malignant cells file is in scratch, not outputs/processed
scrna_path = Path(cfg['datasets']['luad_mhc2_classified']['processed'])
pathway_annot_path = repo_root / cfg['datasets']['cosmx']['pathway_annotations']

# outputs
fig_dir   = repo_root / cfg['outputs']['figures']
table_dir = repo_root / cfg['outputs']['tables']

fig_out   = fig_dir   / 'figure3'
supp_out  = fig_dir   / 'figure3-supp'
table_out = table_dir / 'figure3'

fig_out.mkdir(parents=True, exist_ok=True)
supp_out.mkdir(parents=True, exist_ok=True)
table_out.mkdir(parents=True, exist_ok=True)

## Load data

Malignant cells from the Salcher atlas with MHC II high/low classification are
loaded. The var index is set to gene symbols (`feature_name`) to enable gene
set matching against Hallmark and CosMx pathway collections.

In [ ]:
import warnings

# load scRNA malignant cells with MHC II classification
adata = ad.read_h5ad(scrna_path)
print(f"scRNA malignant cells: {adata.n_obs:,} × {adata.n_vars:,} genes")
print(adata.obs['MHC2_clustering'].value_counts())

# set var index to gene symbols for gene set matching
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    adata.var_names = adata.var['feature_name'].values

print(f"\nVar index (first 5): {adata.var_names[:5].tolist()}")

## Load gene sets

Same two gene set collections as the CosMx analysis — Hallmark and NanoString
CosMx pathway annotations — enabling direct comparison of enrichment results
across modalities.

In [ ]:
# load NanoString CosMx pathway annotations
pathways = pd.read_csv(pathway_annot_path, index_col=0)

long_df = (
    pathways.reset_index()
    .rename(columns={'Gene': 'genesymbol'})
    .melt(id_vars='genesymbol', var_name='geneset', value_name='Belongs')
    .query('Belongs == 1')
    .drop(columns='Belongs')
    .drop_duplicates(subset=['genesymbol', 'geneset'])
    .reset_index(drop=True)
)
long_df['collection'] = 'CosMx'
long_df['geneset'] = 'CosMx: ' + long_df['geneset'].astype(str)
print(f"CosMx pathway sets: {long_df['geneset'].nunique()}")

# load Hallmark gene sets
hallmark = dc.op.hallmark(organism='human', license='academic', verbose=False)
hallmark = hallmark[~hallmark.duplicated(['source', 'target'])]
print(f"Hallmark gene sets: {hallmark['source'].nunique()}")

## Subset to MHC II high vs low

Cells are subset to MHC II high and low classifications. The clonal group
is excluded from this comparison as in the CosMx analysis.

In [ ]:
high_low = adata[
    adata.obs['MHC2_clustering'].isin(['MHC class II High', 'MHC class II Low'])
].copy()
print(f"High + low: {high_low.n_obs:,} cells")
print(high_low.obs['MHC2_clustering'].value_counts())

## GSEA helper functions

Same functions as in `cancer_cell_pathway_analysis` (CosMx) — `run_gsea_collection`
runs `dc.mt.gsea` and stores results in `adata.obsm`; `extract_gsea_results`
aggregates per-cell scores to group level using Mann-Whitney U test.

In [ ]:
def run_gsea_collection(adata, net_df, collection_name, times=200, min_genes_in_set=5):
    """Run dc.mt.gsea — results stored in adata.obsm['score_gsea']."""
    if 'geneset' in net_df.columns:
        net = net_df.rename(columns={'geneset': 'source', 'genesymbol': 'target'})
    else:
        net = net_df.copy()

    net = net[~net.duplicated(['source', 'target'])]
    valid_sets = (
        net.groupby('source')['target']
        .nunique()
        .loc[lambda x: x >= min_genes_in_set]
        .index
    )
    net = net[net['source'].isin(valid_sets)]
    print(f"Running GSEA for {collection_name}: {net['source'].nunique()} gene sets")

    dc.mt.gsea(
        adata, net=net,
        tmin=min_genes_in_set,
        times=times, seed=1, verbose=False,
    )
    print(f"GSEA complete — results stored in adata.obsm['score_gsea']")


def extract_gsea_results(adata, classification_col='MHC2_clustering',
                          pos_label='MHC class II High', neg_label='MHC class II Low'):
    """Aggregate per-cell GSEA scores to group level via Mann-Whitney U."""
    score_df = pd.DataFrame(
        adata.obsm['score_gsea'],
        index=adata.obs_names,
    )
    score_df['group'] = adata.obs[classification_col].values

    results = []
    for pathway in score_df.columns[:-1]:
        pos = score_df.loc[score_df['group'] == pos_label, pathway]
        neg = score_df.loc[score_df['group'] == neg_label, pathway]

        stat, pval = mannwhitneyu(pos, neg, alternative='two-sided')
        mean_pos = pos.mean()
        mean_neg = neg.mean()

        results.append({
            'pathway':  pathway,
            'mean_pos': mean_pos,
            'mean_neg': mean_neg,
            'log2fc':   np.log2(mean_pos / mean_neg),
            'pval':     pval,
        })

    df = pd.DataFrame(results).set_index('pathway')
    df['padj'] = multipletests(df['pval'], method='fdr_bh')[1]
    df['-log10_padj'] = -np.log10(df['padj'])
    return df

## Run GSEA

GSEA is run for Hallmark and CosMx pathway collections. Results are saved
as checkpoints — subsequent runs load from CSV rather than recomputing.
Note: Hallmark must be extracted before running CosMx since both overwrite
`adata.obsm['score_gsea']`.

In [ ]:
%%time

hallmark_results_path = table_out / 'scrna_gsea_hallmark_single_cell.csv'
cosmx_results_path    = table_out / 'scrna_gsea_cosmx_pathways_single_cell.csv'

# --- Hallmark ---
if hallmark_results_path.exists():
    print("Loading pre-computed Hallmark GSEA results...")
    gsea_hallmark = pd.read_csv(hallmark_results_path, index_col=0)
else:
    print("Running Hallmark GSEA...")
    run_gsea_collection(high_low, hallmark, 'hallmark', times=200, min_genes_in_set=5)
    gsea_hallmark = extract_gsea_results(high_low)
    gsea_hallmark.to_csv(hallmark_results_path)
    print(f"Saved → {hallmark_results_path}")

# --- CosMx pathways ---
if cosmx_results_path.exists():
    print("Loading pre-computed CosMx pathway GSEA results...")
    gsea_cosmx = pd.read_csv(cosmx_results_path, index_col=0)
else:
    print("Running CosMx pathway GSEA...")
    run_gsea_collection(high_low, long_df, 'CosMx', times=200, min_genes_in_set=5)
    gsea_cosmx = extract_gsea_results(high_low)
    gsea_cosmx.to_csv(cosmx_results_path)
    print(f"Saved → {cosmx_results_path}")

print("\nTop Hallmark pathways:")
print(gsea_hallmark.sort_values('padj').head(10))

## Visualization

Pathway enrichment results are visualized as a barplot ranked by absolute NES
and a volcano plot of NES vs -log10(FDR). Both plots use the paper color palette
— orange for MHC II high/positive, purple for MHC II low/negative.

In [ ]:
def plot_gsea_barplot(gsea_df, top_n=25, title=None, figsize=(10, 10)):
    """Bar plot of top pathways ranked by absolute NES or log2fc, colored by direction."""
    
    # handle both t-stat GSEA (NES) and per-cell scoring (log2fc)
    score_col = 'NES' if 'NES' in gsea_df.columns else 'log2fc'
    
    plot_df = (
        gsea_df.copy()
        .assign(Direction=np.where(gsea_df[score_col] >= 0, 'MHC-II High', 'MHC-II Low'))
        .assign(absScore=lambda d: d[score_col].abs())
        .sort_values('absScore', ascending=False)
        .head(top_n)
        .sort_values(score_col)
    )
    plot_df.index = plot_df.index.str.replace('_', ' ').str.title()

    fig, ax = plt.subplots(figsize=figsize)
    sns.barplot(
        data=plot_df, y=plot_df.index, x=score_col,
        hue='Direction', dodge=False,
        palette={'MHC-II High': cmap_high_low[0], 'MHC-II Low': cmap_high_low[1]},
        edgecolor='none', ax=ax,
    )
    ax.axvline(0, color='black', lw=1)
    xlabel = 'Normalized Enrichment Score (NES)' if score_col == 'NES' else 'Log2 Fold Change'
    ax.set_xlabel(xlabel)
    ax.set_ylabel('')
    ax.set_title(title or f'Top {top_n} Pathways by |{score_col}|')
    ax.legend(title='Enriched In', bbox_to_anchor=(1.05, 1), loc='upper left')
    sns.despine()
    plt.tight_layout()
    return fig


def plot_gsea_volcano(gsea_df, sig_thresh=0.05, nes_thresh=1.5,
                      top_n_labels=15, title=None, figsize=(10, 8), ymax=30):
    
    score_col = 'NES' if 'NES' in gsea_df.columns else 'log2fc'
    
    df = gsea_df.copy()
    df['-log10_FDR'] = -np.log10(df['padj'].replace(0, np.nextafter(0, 1)))
    df['-log10_FDR'] = df['-log10_FDR'].clip(upper=ymax)
    df['Direction'] = np.where(df[score_col] >= 0, 'MHC-II High', 'MHC-II Low')

    fig, ax = plt.subplots(figsize=figsize)
    sns.scatterplot(
        data=df, x=score_col, y='-log10_FDR', hue='Direction',
        palette={'MHC-II High': cmap_high_low[0], 'MHC-II Low': cmap_high_low[1]},
        edgecolor='none', alpha=0.9, ax=ax,
    )
    ax.axvline( nes_thresh, color='black', lw=1, ls='--')
    ax.axvline(-nes_thresh, color='black', lw=1, ls='--')
    ax.axhline(-np.log10(sig_thresh), color='black', lw=1, ls='--')
    ax.text(0.02, 0.98, f'Note: -log10(FDR) capped at {ymax} for display',
            transform=ax.transAxes, fontsize=8, va='top', color='gray')

    top_hits = df.sort_values('padj').head(top_n_labels)
    texts = []
    for _, row in top_hits.iterrows():
        texts.append(ax.text(
            row[score_col], row['-log10_FDR'],
            row.name.replace('_', ' ').title(),
            fontsize=8,
            ha='right' if row[score_col] < 0 else 'left',
        ))
    adjust_text(texts, arrowprops=dict(arrowstyle='-', color='gray', lw=0.5), ax=ax)

    xlabel = 'Normalized Enrichment Score (NES)' if score_col == 'NES' else 'Log2 Fold Change'
    ax.set_xlabel(xlabel)
    ax.set_ylabel(r'$-\log_{10}$(FDR)')
    ax.set_title(title or 'GSEA Volcano')
    ax.legend(title='Enriched In', bbox_to_anchor=(1.05, 1), loc='upper left')
    sns.despine()
    plt.tight_layout()
    return fig

In [ ]:
# Hallmark barplot
fig = plot_gsea_barplot(
    gsea_hallmark, top_n=25,
    title='Hallmark Pathways — MHC II pos vs neg (Salcher SCRNA)',
    figsize=(7, 10),
)
fig.savefig(supp_out / 'figS3_salcher_hallmark_gsea_barplot_single_cell.pdf', bbox_inches='tight')
plt.show()

# Hallmark volcano
fig = plot_gsea_volcano(
    gsea_hallmark,
    title='Hallmark GSEA Volcano — MHC II pos vs neg (Salcher SCRNA)',
)
fig.savefig(supp_out / 'figS3_salcher_hallmark_gsea_volcano.pdf', bbox_inches='tight')
plt.show()

# CosMx pathways barplot
fig = plot_gsea_barplot(
    gsea_cosmx, top_n=25,
    title='CosMx Pathways — MHC II pos vs neg (Salcher SCRNA)',
    figsize=(7, 10),
)
fig.savefig(supp_out / 'figS3_salcher_cosmx_pathways_gsea_barplot_single_cell.pdf', bbox_inches='tight')
plt.show()

# CosMx pathways volcano
fig = plot_gsea_volcano(
    gsea_cosmx,
    title='CosMx Pathways Volcano — MHC II pos vs neg (Salcher SCRNA)',
)
fig.savefig(supp_out / 'figS3_salcher_cosmx_pathways_gsea_volcano_ttest.pdf', bbox_inches='tight')
plt.show()